<a href="https://colab.research.google.com/github/MayerT1/Prep_GEDI/blob/main/Stage_2_Bake_Off.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

# Define the base project folder and subdirectories
base_dir = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project'
subdirs = [
    '2024_Imagery_For_Inference'
    'data_32_32_patches_11_4_25',
    'target_data',
    'data',
    'data_NaN_filtered',
    'models',
    "model_animations",
    'scripts',
    'notebooks',
    'config',
    'results',
    'prediction_surface'
]

# Create each subdirectory
for subdir in subdirs:
    path = os.path.join(base_dir, subdir)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inferencedata_32_32_patches_11_4_25
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/target_data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data_NaN_filtered
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/models
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/model_animations
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/scripts
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/notebooks
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/config
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/results
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/prediction_surface


In [ ]:
# =====================================================
# STAGE 2 MODEL BAKE-OFF - WITH HYPERPARAMETER TUNING
# Fair comparison with grid search for all models
# =====================================================

import os, sys, time, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import product

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except RuntimeError as e:
    if "TORCH_LIBRARY" in str(e):
        print("⚠️  RESTART RUNTIME"); sys.exit(1)
    raise

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import r2_score
from scipy import stats
# from scipy.spatial.distance import wasserstein_distance
from scipy.stats import wasserstein_distance


print("="*70)
print("STAGE 2 MODEL BAKE-OFF - WITH HYPERPARAMETER TUNING")
print("="*70)

# =====================================================
# CONFIGURATION
# =====================================================

DATA_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Embedding_Checkpoints_Stage1"
OUTPUT_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Tuning strategy
TUNING_METRIC = "coverage"  # Options: "mae", "rmse", "coverage", "composite"
COMPOSITE_WEIGHTS = {
    "mae": 0.4,      # 40% accuracy
    "coverage": 0.4,  # 40% variance preservation
    "r2": 0.2        # 20% fit quality
}

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# PRE-FLIGHT CHECKS
# =====================================================

print("\n🔍 PRE-FLIGHT CHECKS")
print("="*70)

TRAIN_FILE = f"{DATA_DIR}/dataset_train.pt"
TEST_FILE = f"{DATA_DIR}/dataset_test.pt"
VAL_FILE = f"{DATA_DIR}/dataset_val.pt"

checks_passed = True
for name, path in [("Train", TRAIN_FILE), ("Test", TEST_FILE), ("Val", VAL_FILE)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"   ✓ {name}: {size_mb:.1f} MB")
    else:
        print(f"   ✗ {name} NOT FOUND: {path}")
        checks_passed = False

print(f"   ✓ Device: {DEVICE}")
print(f"   ✓ Output: {OUTPUT_DIR}")
print(f"   ✓ Tuning metric: {TUNING_METRIC}")

if not checks_passed:
    print("\n❌ PRE-FLIGHT CHECKS FAILED")
    sys.exit(1)

print("\n✅ ALL CHECKS PASSED")
print("\n⚠️  This will perform hyperparameter tuning + full training.")
print("    Estimated time: 2-4 hours")
print("="*70)

# =====================================================
# LOAD DATA
# =====================================================

print("\nLOADING DATA")
print("="*70)

dataset_train = torch.load(TRAIN_FILE, weights_only=False)
dataset_test = torch.load(TEST_FILE, weights_only=False)
dataset_val = torch.load(VAL_FILE, weights_only=False)

def extract_data(dataset):
    X, y = [], []
    for s in dataset:
        X.append(s['embeddings'].flatten().numpy())
        y.append(s['target'].item())
    return np.array(X), np.array(y)

X_train, y_train = extract_data(dataset_train)
X_test, y_test = extract_data(dataset_test)
X_val, y_val = extract_data(dataset_val)

print(f"✓ Train: {X_train.shape} samples")
print(f"✓ Val:   {X_val.shape} samples")
print(f"✓ Test:  {X_test.shape} samples")
print(f"\n✓ Target ranges:")
print(f"  Train: {y_train.min():.1f}-{y_train.max():.1f}m (std={y_train.std():.1f}m)")
print(f"  Val:   {y_val.min():.1f}-{y_val.max():.1f}m (std={y_val.std():.1f}m)")
print(f"  Test:  {y_test.min():.1f}-{y_test.max():.1f}m (std={y_test.std():.1f}m)")

target_range_val = y_val.max() - y_val.min()
target_range_test = y_test.max() - y_test.min()

# =====================================================
# HYPERPARAMETER GRIDS
# =====================================================

print("\n" + "="*70)
print("DEFINING HYPERPARAMETER SEARCH SPACES")
print("="*70)

HYPERPARAMETER_GRIDS = {
    "RF_Raw_Baseline": {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },

    "Ridge": {
        "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
    },

    "Lasso": {
        "alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
    },

    "ElasticNet": {
        "alpha": [0.1, 1.0, 10.0],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    },

    "RF_Embeddings": {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },

    "SimpleMLP": {
        "hidden_dims": [
            [2048, 512, 128],
            [2048, 1024, 512, 128],
            [4096, 1024, 256]
        ],
        "dropout": [0.2, 0.3, 0.4],
        "lr": [5e-4, 1e-3, 2e-3]
    },

    "MoE": {
        "num_experts": [3, 5, 7],
        "hidden_dims": [[1024, 512, 256], [2048, 512, 256]],
        "dropout": [0.2, 0.3],
        "lr": [5e-4, 1e-3]
    },

    "Hierarchical": {
        "num_layers": [2, 4, 6],
        "nhead": [4, 8],
        "dropout": [0.1, 0.2, 0.3],
        "lr": [5e-4, 1e-3]
    }
}

# Print grid sizes
print("\nSearch space sizes:")
for model_name, grid in HYPERPARAMETER_GRIDS.items():
    param_grid = ParameterGrid(grid)
    print(f"  {model_name}: {len(param_grid)} combinations")

total_configs = sum(len(ParameterGrid(grid)) for grid in HYPERPARAMETER_GRIDS.values())
print(f"\n✓ Total configurations to evaluate: {total_configs}")

# =====================================================
# PYTORCH COMPONENTS
# =====================================================

class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# MODEL ARCHITECTURES (parameterized)

class SimpleMLP(nn.Module):
    def __init__(self, hidden_dims=[2048, 512, 128], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = 65536
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

class MixtureOfExperts(nn.Module):
    def __init__(self, num_experts=5, hidden_dims=[1024, 512, 256], dropout=0.3):
        super().__init__()
        self.num_experts = num_experts

        self.encoder = nn.Sequential(
            nn.Linear(65536, 2048),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.experts = nn.ModuleList([
            self._build_expert(1024, hidden_dims, dropout)
            for _ in range(num_experts)
        ])

        self.gate = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, num_experts),
            nn.Softmax(dim=-1)
        )

    def _build_expert(self, input_dim, hidden_dims, dropout):
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        feat = self.encoder(x)
        exp_preds = torch.stack([e(feat).squeeze(-1) for e in self.experts], dim=-1)
        weights = self.gate(feat)
        return (exp_preds * weights).sum(dim=-1)

class HierarchicalFusion(nn.Module):
    def __init__(self, num_layers=4, nhead=4, dropout=0.2):
        super().__init__()
        self.mod_emb = nn.Embedding(8, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=nhead,
            dim_feedforward=1024,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        B = x.shape[0]
        mod_tokens = x.reshape(B, 8, 64, 128)
        mod_mean = mod_tokens.mean(dim=2)
        mod_max = mod_tokens.max(dim=2)[0]
        mod_embeds = torch.cat([mod_mean, mod_max], dim=-1)

        mod_ids = torch.arange(8, device=x.device)
        type_emb = self.mod_emb(mod_ids).unsqueeze(0).expand(B, -1, -1).repeat(1, 1, 2)
        mod_embeds = mod_embeds + type_emb

        fused = self.transformer(mod_embeds)
        return self.head(fused.mean(dim=1)).squeeze(-1)

# =====================================================
# EVALUATION FUNCTIONS
# =====================================================

def evaluate_predictions(y_true, y_pred, target_range):
    """Compute all metrics for a set of predictions"""
    mae = np.abs(y_pred - y_true).mean()
    rmse = np.sqrt(((y_pred - y_true)**2).mean())
    r2 = r2_score(y_true, y_pred)

    pred_range = y_pred.max() - y_pred.min()
    coverage = (pred_range / target_range) * 100
    std_ratio = y_pred.std() / y_true.std()

    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'coverage': coverage,
        'std_ratio': std_ratio,
        'pred_range': pred_range
    }

def compute_composite_score(metrics):
    """Compute weighted composite score (lower is better for MAE/RMSE)"""
    # Normalize metrics to [0, 1] where lower is better
    mae_norm = metrics['mae'] / 20.0  # Assume max MAE ~ 20m
    coverage_norm = 1.0 - (metrics['coverage'] / 100.0)  # Invert coverage
    r2_norm = 1.0 - max(0, metrics['r2'])  # Invert R2

    score = (COMPOSITE_WEIGHTS['mae'] * mae_norm +
             COMPOSITE_WEIGHTS['coverage'] * coverage_norm +
             COMPOSITE_WEIGHTS['r2'] * r2_norm)

    return score

def select_best_config(results, metric):
    """Select best hyperparameter configuration"""
    if metric == "mae":
        return min(results, key=lambda x: x['metrics']['mae'])
    elif metric == "rmse":
        return min(results, key=lambda x: x['metrics']['rmse'])
    elif metric == "coverage":
        return max(results, key=lambda x: x['metrics']['coverage'])
    elif metric == "composite":
        return min(results, key=lambda x: compute_composite_score(x['metrics']))
    else:
        raise ValueError(f"Unknown metric: {metric}")

# =====================================================
# TRAINING FUNCTIONS
# =====================================================

def train_pytorch_model(model, train_loader, val_loader, lr=1e-3, epochs=100, patience=25):
    """Train a PyTorch model with early stopping"""
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.L1Loss()

    best_val_mae = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Train
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validate
        model.eval()
        val_preds = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                preds = model(X_batch)
                val_preds.append(preds.cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_mae = np.abs(val_preds - y_val).mean()

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best model
    model.load_state_dict(best_model_state)
    return model

# =====================================================
# HYPERPARAMETER TUNING - SKLEARN MODELS
# =====================================================

print("\n" + "="*70)
print("PHASE 1: HYPERPARAMETER TUNING (Validation Set)")
print("="*70)

tuning_results = {}

# 1. RF Raw Baseline
print("\n[1/8] Tuning RF_Raw_Baseline...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["RF_Raw_Baseline"])
results = []
for i, params in enumerate(param_grid):
    if (i+1) % 10 == 0:
        print(f"  Config {i+1}/{len(param_grid)}")

    rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred_val = rf.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["RF_Raw_Baseline"] = best_config
print(f"  ✓ Best: {best_config['params']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 2. Ridge
print("\n[2/8] Tuning Ridge...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Ridge"])
results = []
for params in param_grid:
    ridge = Ridge(**params)
    ridge.fit(X_train, y_train)
    pred_val = ridge.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["Ridge"] = best_config
print(f"  ✓ Best: {best_config['params']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 3. Lasso
print("\n[3/8] Tuning Lasso...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Lasso"])
results = []
for params in param_grid:
    lasso = Lasso(**params, max_iter=5000)
    lasso.fit(X_train, y_train)
    pred_val = lasso.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["Lasso"] = best_config
print(f"  ✓ Best: {best_config['params']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 4. ElasticNet
print("\n[4/8] Tuning ElasticNet...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["ElasticNet"])
results = []
for params in param_grid:
    enet = ElasticNet(**params, max_iter=5000)
    enet.fit(X_train, y_train)
    pred_val = enet.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["ElasticNet"] = best_config
print(f"  ✓ Best: {best_config['params']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 5. RF Embeddings
print("\n[5/8] Tuning RF_Embeddings...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["RF_Embeddings"])
results = []
for i, params in enumerate(param_grid):
    if (i+1) % 10 == 0:
        print(f"  Config {i+1}/{len(param_grid)}")

    rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred_val = rf.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["RF_Embeddings"] = best_config
print(f"  ✓ Best: {best_config['params']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# =====================================================
# HYPERPARAMETER TUNING - PYTORCH MODELS
# =====================================================

train_loader = DataLoader(SimpleDataset(X_train, y_train), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(SimpleDataset(X_val, y_val), batch_size=32, shuffle=False, num_workers=0)

# 6. SimpleMLP
print("\n[6/8] Tuning SimpleMLP...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["SimpleMLP"])
results = []
for i, params in enumerate(param_grid):
    print(f"  Config {i+1}/{len(param_grid)}: hidden={params['hidden_dims']}, lr={params['lr']}, dropout={params['dropout']}")

    model = SimpleMLP(hidden_dims=params['hidden_dims'], dropout=params['dropout'])
    model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

    model.eval()
    with torch.no_grad():
        pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

    print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["SimpleMLP"] = best_config
print(f"  ✓ Best: hidden={best_config['params']['hidden_dims']}, lr={best_config['params']['lr']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 7. MoE
print("\n[7/8] Tuning MoE...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["MoE"])
results = []
for i, params in enumerate(param_grid):
    print(f"  Config {i+1}/{len(param_grid)}: experts={params['num_experts']}, lr={params['lr']}")

    model = MixtureOfExperts(
        num_experts=params['num_experts'],
        hidden_dims=params['hidden_dims'],
        dropout=params['dropout']
    )
    model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

    model.eval()
    with torch.no_grad():
        pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

    print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["MoE"] = best_config
print(f"  ✓ Best: experts={best_config['params']['num_experts']}, lr={best_config['params']['lr']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# 8. Hierarchical
print("\n[8/8] Tuning Hierarchical...")
param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Hierarchical"])
results = []
for i, params in enumerate(param_grid):
    print(f"  Config {i+1}/{len(param_grid)}: layers={params['num_layers']}, nhead={params['nhead']}, lr={params['lr']}")

    model = HierarchicalFusion(
        num_layers=params['num_layers'],
        nhead=params['nhead'],
        dropout=params['dropout']
    )
    model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

    model.eval()
    with torch.no_grad():
        pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

    metrics = evaluate_predictions(y_val, pred_val, target_range_val)
    results.append({'params': params, 'metrics': metrics})

    print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

best_config = select_best_config(results, TUNING_METRIC)
tuning_results["Hierarchical"] = best_config
print(f"  ✓ Best: layers={best_config['params']['num_layers']}, nhead={best_config['params']['nhead']}")
print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# =====================================================
# SAVE TUNING RESULTS
# =====================================================

print("\n" + "="*70)
print("TUNING COMPLETE - SAVING RESULTS")
print("="*70)

tuning_summary = []
for model_name, config in tuning_results.items():
    tuning_summary.append({
        'Model': model_name,
        'Best_Params': str(config['params']),
        'Val_MAE': config['metrics']['mae'],
        'Val_Coverage': config['metrics']['coverage'],
        'Val_RMSE': config['metrics']['rmse'],
        'Val_R2': config['metrics']['r2']
    })

df_tuning = pd.DataFrame(tuning_summary)
tuning_path = os.path.join(OUTPUT_DIR, "Stage2_Bakeoff_Tuning_Results.csv")
df_tuning.to_csv(tuning_path, index=False)
print(f"\n✓ Saved tuning results: {tuning_path}")

print("\n" + "="*70)
print("BEST HYPERPARAMETERS SELECTED")
print("="*70)
print(df_tuning.to_string(index=False))

# Save full tuning results as pickle
tuning_pickle_path = os.path.join(OUTPUT_DIR, "Stage2_Bakeoff_Tuning_Full.pkl")
with open(tuning_pickle_path, 'wb') as f:
    pickle.dump(tuning_results, f)
print(f"\n✓ Saved full tuning data: {tuning_pickle_path}")

print("\n" + "="*70)
print("PHASE 1 COMPLETE - Ready for Phase 2 (Final Training)")
print("="*70)
print("\nNext: Run Phase 2 script to train models with best hyperparameters on test set")

STAGE 2 MODEL BAKE-OFF - WITH HYPERPARAMETER TUNING

🔍 PRE-FLIGHT CHECKS
   ✓ Train: 170.0 MB
   ✓ Test: 38.1 MB
   ✓ Val: 21.5 MB
   ✓ Device: cpu
   ✓ Output: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results
   ✓ Tuning metric: coverage

✅ ALL CHECKS PASSED

⚠️  This will perform hyperparameter tuning + full training.
    Estimated time: 2-4 hours

LOADING DATA
✓ Train: (679, 65536) samples
✓ Val:   (86, 65536) samples
✓ Test:  (152, 65536) samples

✓ Target ranges:
  Train: -1.1-23.5m (std=7.0m)
  Val:   -0.4-26.2m (std=7.3m)
  Test:  -0.4-24.7m (std=6.9m)

DEFINING HYPERPARAMETER SEARCH SPACES

Search space sizes:
  RF_Raw_Baseline: 108 combinations
  Ridge: 5 combinations
  Lasso: 5 combinations
  ElasticNet: 15 combinations
  RF_Embeddings: 108 combinations
  SimpleMLP: 27 combinations
  MoE: 24 combinations
  Hierarchical: 36 combinations

✓ Total configurations to evaluate: 328

PHASE 1: HYPERPARAMETER TUNING (Validation Set)

[1/8] Tuni